# 환경설정


In [1]:
%pip install -q langchain langchain-openai python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# 현재 환경 확인
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")


✅ 로컬 Jupyter


In [3]:
import os
import json
from pathlib import Path
from datetime import datetime
from getpass import getpass
from dotenv import load_dotenv
# langchain 패키지 가져오기
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 현재 작업 폴더 기준
BASE_DIR = Path.cwd()
print("현재 작업 폴더:", BASE_DIR)
print("✅ import 완료")


현재 작업 폴더: c:\workspaces\chatbot\langchain-web-chatbot
✅ import 완료


In [4]:
# OPENAI API 키 확인
load_dotenv(BASE_DIR / ".env")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API 키: ").strip()
print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")


✅ API 키 OK


### 실습과제 2번


In [5]:
# 페르소나 사전: 이름 → system 메시지
PERSONAS = {
    "분석가": "당신은 10년 차 데이터 분석가입니다. 수치와 근거를 들어 간결하게 답변하세요.",
    "마케터": "당신은 디지털 마케팅 전문가입니다. 비즈니스 관점과 사례를 들어 답변하세요.",
    "개발자": "당신은 시니어 백엔드 개발자입니다. 코드 예시와 함께 실용적으로 답변하세요.",
    "기획자": "당신은 IT 서비스 기획자입니다. 사용자 가치와 우선순위 관점에서 답변하세요.",
    "디자이너": "당신은 UX 디자이너입니다. 사용자 경험과 시각적 비유로 쉽게 답변하세요.",
}

In [6]:
# 페르소사 소개 사전
PERSONA_INTROS = {
    "분석가": "수치와 근거 기반 답변",
    "마케터": "비즈니스 관점과 마케팅 사례 중심 답변",
    "개발자": "코드 예시를 활용한 실용적인 답변",
    "기획자": "사용자 가치와 우선순위 중심 답변",
    "디자이너": "사용자 경험과 시각적 비유로 쉬운 답변",
}

In [7]:
# 페르소나 temperature 설정 블럭
TEMPERATURES = {
    "분석가": 0,
    "마케터": 0.5,
    "개발자": 0.3,
    "기획자": 0.9,
    "디자이너": 0.8,
}

# 페르소사 소개 사전
PERSONA_INTROS = {
    "분석가": "수치와 근거 기반 답변",
    "마케터": "비즈니스 관점과 마케팅 사례 중심 답변",
    "개발자": "코드 예시를 활용한 실용적인 답변",
    "기획자": "사용자 가치와 우선순위 중심 답변",
    "디자이너": "사용자 경험과 시각적 비유로 쉬운 답변",
}

In [8]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

prompt = ChatPromptTemplate.from_messages([
    ("system", "{role}"),
    ("human", "{question}"),
])

# llm은 미리 생성한 LangChain 채팅 모델
PERSONA_LLMS = {}

for name in PERSONAS:
    PERSONA_LLMS[name] = llm.bind(temperature=TEMPERATURES.get(name, 0.3))


In [9]:
for name, persona_llm in PERSONA_LLMS.items():
    print(f"{name}: temperature={persona_llm.kwargs['temperature']}")


분석가: temperature=0
마케터: temperature=0.5
개발자: temperature=0.3
기획자: temperature=0.9
디자이너: temperature=0.8


## 실행


In [ ]:
while True:
    print("선택 가능:", ", ".join(PERSONAS))
    persona = input("페르소나 선택 (종료: 종료): ").strip()

    if persona in ("종료", "/quit", "/exit"):
        break

    if persona not in PERSONAS:
        print("목록에 있는 페르소나를 선택하세요.")
        continue

    user_input = input("질문: ").strip()

    if not user_input:
        continue

    persona_llm = PERSONA_LLMS[persona]
    chain = prompt | persona_llm | StrOutputParser()

    answer = chain.invoke({
        "role": PERSONAS[persona],
        "question": user_input,
    })

    print(answer)


선택 가능: 분석가, 마케터, 개발자, 기획자, 디자이너
안녕하세요! 어떻게 도와드릴까요? 백엔드 개발 관련 질문이나 코드 예시가 필요하시면 말씀해 주세요.
선택 가능: 분석가, 마케터, 개발자, 기획자, 디자이너
